In [1]:
import dagshub
import mlflow

# Connect to your DagsHub repo
dagshub.init(repo_owner="shreychauhan02", repo_name="my_final_project", mlflow=True)



Accessing as shreychauhan02

Initialized MLflow to track repo "shreychauhan02/my_final_project"

Repository shreychauhan02/my_final_project initialized!

In [2]:
# Set or create an experiment
mlflow.set_experiment("TfIdf Trigram max_features")

2026/09/25 10:11:18 INFO mlflow.tracking.fluent: Experiment with name 'TfIdf Trigram max_features' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/42223e5d58ec49d19b275a4ba1e0a814', creation_time=1790311279170, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1790311279170, lifecycle_stage='active', name='TfIdf Trigram max_features', tags={}, trace_location=None, workspace='default'>

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

In [4]:
df = pd.read_csv('../dataset/pre.csv').dropna(subset=['clean_comment'])
df.shape

(36662, 2)

In [ ]:
# Step 1: Function to run the experiment
def run_experiment_tfidf_max_features(max_features):
    ngram_range = (1, 3)  # Trigram setting

    vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)

    X_text = df['clean_comment']
    y = df['category']

    # Step 2: Train-test split on raw text FIRST (no leakage from test vocab/IDF)
    X_train, X_test, y_train, y_test = train_test_split(X_text, y, test_size=0.2, random_state=42, stratify=y)

    # Step 3: Fit vectorizer on train only, transform test with fitted vocab
    X_train = vectorizer.fit_transform(X_train)
    X_test = vectorizer.transform(X_test)

    # Step 4: Define and train a Random Forest model
    with mlflow.start_run() as run:
        # Set tags for the experiment and run
        mlflow.set_tag("mlflow.runName", f"TFIDF_Trigrams_max_features_{max_features}")
        mlflow.set_tag("experiment_type", "feature_engineering")
        mlflow.set_tag("model_type", "RandomForestClassifier")

        # Add a description
        mlflow.set_tag("description", f"RandomForest with TF-IDF Trigrams, max_features={max_features}")

        # Log vectorizer parameters
        mlflow.log_param("vectorizer_type", "TF-IDF")
        mlflow.log_param("ngram_range", ngram_range)
        mlflow.log_param("vectorizer_max_features", max_features)

        # Log Random Forest parameters
        n_estimators = 200
        max_depth = 15

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)

        # Initialize and train the model
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train, y_train)

        # Step 5: Make predictions and log metrics
        y_pred = model.predict(X_test)

        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)

        # Log classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                    mlflow.log_metric(f"{label}_{metric}", value)

        # Log confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)
        plt.figure(figsize=(8, 6))
        sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title(f"Confusion Matrix: TF-IDF Trigrams, max_features={max_features}")
        plt.savefig("confusion_matrix.png")
        mlflow.log_artifact("confusion_matrix.png")
        plt.close()

        # Log the model
        mlflow.sklearn.log_model(model, f"random_forest_model_tfidf_trigrams_{max_features}")

# Step 6: Test various max_features values
max_features_values = [1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000]

for max_features in max_features_values:
    run_experiment_tfidf_max_features(max_features)

2026/09/25 10:12:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run TFIDF_Trigrams_max_features_1000 at: https://dagshub.com/shreychauhan02/my_final_project.mlflow/#/experiments/2/runs/50fc84345c4b426d9de8806ac830baa4
🧪 View experiment at: https://dagshub.com/shreychauhan02/my_final_project.mlflow/#/experiments/2


2026/09/25 10:15:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run TFIDF_Trigrams_max_features_2000 at: https://dagshub.com/shreychauhan02/my_final_project.mlflow/#/experiments/2/runs/caae0c917adc482a9d343aa6bfac38c9
🧪 View experiment at: https://dagshub.com/shreychauhan02/my_final_project.mlflow/#/experiments/2


2026/09/25 10:18:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run TFIDF_Trigrams_max_features_3000 at: https://dagshub.com/shreychauhan02/my_final_project.mlflow/#/experiments/2/runs/5be737f46ce94b7986aeb4f9ef90d742
🧪 View experiment at: https://dagshub.com/shreychauhan02/my_final_project.mlflow/#/experiments/2


2026/09/25 10:21:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run TFIDF_Trigrams_max_features_4000 at: https://dagshub.com/shreychauhan02/my_final_project.mlflow/#/experiments/2/runs/1b044cc2f84748af9af4c45043f310a6
🧪 View experiment at: https://dagshub.com/shreychauhan02/my_final_project.mlflow/#/experiments/2


2026/09/25 10:23:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
